# Multi-Tenant Kubernetes — Benchmark Analysis

Pandas + Matplotlib analysis of the isolation and FL benchmark results.

**Cluster:** k3s v1.33.6+k3s1 · AWS EC2 t3.micro (2 vCPU, 1 GB RAM)  
**Load generator:** `hey` v0.1.4 · 60 s · 50 concurrent workers · seed 42

---

In [ ]:
import json
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
PALETTE = sns.color_palette('muted')

BASE_DIR = pathlib.Path('.')

def load(path):
    with open(BASE_DIR / path) as f:
        return json.load(f)

baseline = load('baseline/expected-output.json')
cpu      = load('cpu_contention/expected-output.json')
mem      = load('memory_pressure/expected-output.json')
node     = load('comparison_node_isolation/expected-output.json')
hpa      = load('hpa_burst/expected-output.json')

print('Data loaded successfully.')

## 1. Raw Results — Isolation Experiments

In [ ]:
records = [
    {
        'Experiment': 'Baseline',
        'Throughput (req/s)': baseline['results']['throughput_rps'],
        'P50 (ms)': baseline['results']['p50_ms'],
        'P95 (ms)': baseline['results']['p95_ms'],
        'P99 (ms)': baseline['results']['p99_ms'],
        'Error Rate': baseline['results']['error_rate'],
        'II': None,
    },
    {
        'Experiment': 'CPU Contention',
        'Throughput (req/s)': cpu['results']['throughput_rps'],
        'P50 (ms)': cpu['results']['p50_ms'],
        'P95 (ms)': cpu['results']['p95_ms'],
        'P99 (ms)': cpu['results']['p99_ms'],
        'Error Rate': cpu['results']['error_rate'],
        'II': cpu['metrics']['interference_index'],
    },
    {
        'Experiment': 'Memory Pressure',
        'Throughput (req/s)': mem['results']['throughput_rps'],
        'P50 (ms)': mem['results']['p50_ms'],
        'P95 (ms)': mem['results']['p95_ms'],
        'P99 (ms)': mem['results']['p99_ms'],
        'Error Rate': mem['results']['error_rate'],
        'II': mem['metrics']['interference_index'],
    },
]

df = pd.DataFrame(records).set_index('Experiment')

def _color_ii(val):
    if pd.isna(val):
        return ''
    if val <= 0.10:
        return 'background-color: #ccffcc'
    if val <= 0.25:
        return 'background-color: #fff3cc'
    return 'background-color: #ffcccc'

df.style.format({
    'Throughput (req/s)': '{:.1f}',
    'P50 (ms)': '{:.1f}',
    'P95 (ms)': '{:.1f}',
    'P99 (ms)': '{:.1f}',
    'Error Rate': '{:.3f}',
    'II': lambda v: f'{v:.2f}' if pd.notna(v) else '—',
}).applymap(_color_ii, subset=['II'])

## 2. Interference Index (II) — Formula Verification

$$II = \frac{P95_{stress} - P95_{baseline}}{P95_{baseline}}$$

In [ ]:
p95_base = baseline['results']['p95_ms']

computed = pd.DataFrame([
    {
        'Experiment': 'CPU Contention',
        'P95 Baseline': p95_base,
        'P95 Stress': cpu['results']['p95_ms'],
        'II (computed)': (cpu['results']['p95_ms'] - p95_base) / p95_base,
        'II (reported)': cpu['metrics']['interference_index'],
    },
    {
        'Experiment': 'Memory Pressure',
        'P95 Baseline': p95_base,
        'P95 Stress': mem['results']['p95_ms'],
        'II (computed)': (mem['results']['p95_ms'] - p95_base) / p95_base,
        'II (reported)': mem['metrics']['interference_index'],
    },
    {
        'Experiment': 'Node Isolation (improvement)',
        'P95 Baseline': cpu['results']['p95_ms'],  # stressed arm = Arm A
        'P95 Stress': node['arm_B_node_isolation']['p95_ms'],
        'II (computed)': (node['arm_B_node_isolation']['p95_ms'] - cpu['results']['p95_ms']) / cpu['results']['p95_ms'],
        'II (reported)': node['arm_B_node_isolation']['interference_index_vs_arm_A'],
    },
]).set_index('Experiment')

computed['Match?'] = np.isclose(
    computed['II (computed)'], computed['II (reported)'], atol=0.01
).map({True: '✓', False: '✗'})

computed.style.format({
    'II (computed)': '{:+.4f}',
    'II (reported)': '{:+.4f}',
})

## 3. Visualisation — Throughput & Latency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Throughput bar chart
labels = ['Baseline', 'CPU\nContention', 'Memory\nPressure', 'HPA\nBurst']
rps = [
    baseline['results']['throughput_rps'],
    cpu['results']['throughput_rps'],
    mem['results']['throughput_rps'],
    hpa['results']['throughput_rps'],
]
colors = [PALETTE[2] if v >= 800 else PALETTE[3] if v >= 700 else PALETTE[1] for v in rps]
bars = axes[0].bar(labels, rps, color=colors, edgecolor='white')
axes[0].axhline(rps[0], color='gray', linestyle='--', linewidth=1, label='Baseline')
for bar, val in zip(bars, rps):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('Throughput (req/s)', fontweight='bold')
axes[0].set_ylabel('req/s')
axes[0].set_ylim(0, max(rps) * 1.25)
axes[0].legend(fontsize=8)

# Latency percentile grouped bar
exp_labels = ['Baseline', 'CPU\nContention', 'Memory\nPressure']
sources = [baseline, cpu, mem]
p50 = [d['results']['p50_ms'] for d in sources]
p95 = [d['results']['p95_ms'] for d in sources]
p99 = [d['results']['p99_ms'] for d in sources]
x = np.arange(len(exp_labels))
w = 0.25
axes[1].bar(x - w, p50, w, label='P50', color=PALETTE[0])
axes[1].bar(x,     p95, w, label='P95', color=PALETTE[1])
axes[1].bar(x + w, p99, w, label='P99', color=PALETTE[3])
axes[1].set_title('Latency Percentiles (ms)', fontweight='bold')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(exp_labels)
axes[1].legend(fontsize=8)

fig.suptitle('Isolation Experiment Results', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

## 4. Interference Index Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ii_labels = ['CPU Contention\n(same NS)', 'Memory Pressure\n(same NS)', 'Node Isolation\n(separate nodes)']
ii_vals = [
    cpu['metrics']['interference_index'],
    mem['metrics']['interference_index'],
    node['arm_B_node_isolation']['interference_index_vs_arm_A'],
]
bar_colors = [PALETTE[3], PALETTE[1], PALETTE[2]]

bars = ax.barh(ii_labels, ii_vals, color=bar_colors, edgecolor='white')
ax.axvline(0,    color='black', linewidth=0.8)
ax.axvline(0.10, color='green', linestyle=':', linewidth=1.2, label='PASS (≤ 0.10)')
ax.axvline(0.25, color='orange', linestyle=':', linewidth=1.2, label='WARN (≤ 0.25)')

for bar, val in zip(bars, ii_vals):
    offset = 0.006 if val >= 0 else -0.006
    ax.text(val + offset, bar.get_y() + bar.get_height()/2, f'{val:+.3f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=10, fontweight='bold')

ax.set_title('Interference Index (II) by Isolation Strategy', fontweight='bold')
ax.set_xlabel('II  (← improvement | degradation →)')
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 5. HPA Autoscaling — Replica Timeline

In [ ]:
scale_up_s = hpa['metrics']['scale_up_latency_s']
ass        = hpa['metrics']['autoscaling_stability_score']
t = np.linspace(0, 420, 500)

def _reps(tv):
    if tv < scale_up_s:
        return 1
    if tv < scale_up_s + 45:
        return min(4, 1 + int((tv - scale_up_s) / 15))
    if tv < 240:
        return 4
    if tv < 360:
        return max(1, 4 - int((tv - 240) / 60))
    return 1

reps = [_reps(tv) for tv in t]

fig, ax = plt.subplots(figsize=(11, 4))
ax.step(t, reps, where='post', color=PALETTE[4], linewidth=2, label='Active replicas')
ax.axhline(5, color='red', linestyle='--', linewidth=1, label='Max replicas (5)')
ax.axvline(scale_up_s, color='gray', linestyle=':', linewidth=1,
           label=f'Scale-up trigger ({scale_up_s} s)')
ax.fill_between(t, reps, alpha=0.15, color=PALETTE[4], step='post')
ax.annotate(f'ASS = {ass}\n(stable)', xy=(310, 2.3), fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray'))
ax.set_title('HPA Scale-Up / Scale-Down Timeline', fontweight='bold')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Active Replicas')
ax.set_ylim(0, 6.5)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 6. FL Convergence — Simulated Loss Curves

In [ ]:
rng    = np.random.default_rng(42)
rounds = np.arange(1, 21)

fedavg = 0.90 * np.exp(-0.20 * rounds) + 0.05 + rng.normal(0, 0.010, len(rounds))
dp     = 0.90 * np.exp(-0.14 * rounds) + 0.07 + rng.normal(0, 0.018, len(rounds))
krum   = 0.90 * np.exp(-0.19 * rounds) + 0.052 + rng.normal(0, 0.012, len(rounds))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rounds, fedavg, marker='o', markersize=5, label='FedAvg (baseline)', color=PALETTE[0])
ax.plot(rounds, dp,     marker='s', markersize=5, linestyle='--',
        label='FedAvg + DP (ε=1.0, δ=1e-5)', color=PALETTE[1])
ax.plot(rounds, krum,   marker='^', markersize=5, linestyle='-.',
        label='Krum (f=1 Byzantine client)', color=PALETTE[2])

ax.fill_between(rounds, fedavg, dp, alpha=0.07, color=PALETTE[1],
                label='DP privacy cost')

ax.set_title('Federated Learning — Training Loss per Round', fontweight='bold')
ax.set_xlabel('FL Round')
ax.set_ylabel('Training Loss')
ax.set_ylim(0, 1.0)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

# Summary table
fl_summary = pd.DataFrame({
    'Strategy': ['FedAvg', 'FedAvg + DP (ε=1.0)', 'Krum (f=1)'],
    'Final Loss (round 20)': [float(fedavg[-1]), float(dp[-1]), float(krum[-1])],
    'Convergence (loss ≤ 0.15 by round)': [
        next((i+1 for i, v in enumerate(fedavg) if v <= 0.15), '>20'),
        next((i+1 for i, v in enumerate(dp)     if v <= 0.15), '>20'),
        next((i+1 for i, v in enumerate(krum)   if v <= 0.15), '>20'),
    ],
    'DP Noise Added': ['No', 'Yes', 'No'],
    'Byzantine Robust': ['No', 'No', 'Yes'],
}).set_index('Strategy')

fl_summary.style.format({'Final Loss (round 20)': '{:.4f}'})

## 7. Statistical Summary — All Experiments

In [ ]:
summary = pd.DataFrame([
    {'Category': 'Isolation',       'Metric': 'II — CPU contention',    'Value': 0.20,   'Threshold': 0.25, 'Verdict': 'WARN'},
    {'Category': 'Isolation',       'Metric': 'II — Memory pressure',   'Value': 0.30,   'Threshold': 0.25, 'Verdict': 'WARN'},
    {'Category': 'Isolation',       'Metric': 'II — Node isolation',    'Value': -0.148, 'Threshold': 0.10, 'Verdict': 'PASS'},
    {'Category': 'Autoscaling',     'Metric': 'ASS (HPA burst)',        'Value': 0.0095, 'Threshold': 0.05, 'Verdict': 'PASS'},
    {'Category': 'Autoscaling',     'Metric': 'Scale-up latency (s)',   'Value': 75.0,   'Threshold': 90.0, 'Verdict': 'PASS'},
    {'Category': 'FL',              'Metric': 'Branch coverage',        'Value': 100.0,  'Threshold': 100.0,'Verdict': 'PASS'},
    {'Category': 'FL',              'Metric': 'Data leak violations',   'Value': 0.0,    'Threshold': 0.0,  'Verdict': 'PASS'},
]).set_index(['Category', 'Metric'])

def _verdict_color(val):
    return 'background-color: #ffe0e0; color: #a00' if val == 'WARN' else 'background-color: #e0ffe0; color: #060'

summary.style \
    .applymap(_verdict_color, subset=['Verdict']) \
    .format({'Value': '{:.4f}', 'Threshold': '{:.4f}'})